# 01 — Ingestion Pipeline

**Learning objectives:**
- Load documents from multiple sources
- Apply chunking strategies
- Embed and index into ChromaDB

## Setup

Import modules and print configuration.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import FALLBACK_DIR, print_config
print_config()

## Load Documents

Load the bundled SQuAD fallback corpus.

In [ ]:
# ── EXERCISE ──────────────────────────────────────────────────────────────
# Load the SQuAD fallback corpus and print:
#   - Number of documents loaded
#   - First document's text (first 200 chars)
#   - First document's metadata

# YOUR CODE HERE
documents = ???

print(f"Loaded: {???} documents")
print(f"First doc preview: {???}")
print(f"Metadata: {???}")

In [ ]:
# ── SOLUTION  ──────────────────────────────────────────────
from src.ingestion.loader import load_documents
from src.config import FALLBACK_DIR

documents = load_documents(FALLBACK_DIR / "squad_sample.json")

print(f"Loaded: {len(documents)} documents")
print(f"First doc preview: {documents[0].text[:200]}")
print(f"Metadata: {documents[0].metadata}")

## Chunk Documents

Split documents into retrieval-sized chunks.

In [ ]:
# ── EXERCISE ──────────────────────────────────────────────────────────────
# Chunk documents using the 'sentence' strategy.
# Print the number of nodes created.

# Note: chunk_documents() does NOT need embed_model for 'sentence' strategy.
# embed_model is configured in the next cell for the index step.

# YOUR CODE HERE
from src.ingestion.chunker import chunk_documents

nodes = ???
print(f"Created {???} nodes from {len(documents)} documents")
print(f"\nFirst node preview:\n{nodes[0].get_content()[:200]}")

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────
from src.ingestion.chunker import chunk_documents

nodes = chunk_documents(documents, strategy="sentence")
print(f"Created {len(nodes)} nodes from {len(documents)} documents")
print(f"\nFirst node preview:\n{nodes[0].get_content()[:200]}")

## Step 3 — Configure Embedding Model

The embedding model converts text chunks into vectors.

- **With `GEMINI_API_KEY`**: uses `text-embedding-004` (API, higher quality)
- **Without any key**: uses `all-MiniLM-L6-v2` (local CPU, always works)

`configure_embed_model()` picks the best available option automatically.

In [ ]:
from src.ingestion.embedder import configure_embed_model

embed_model = configure_embed_model()
print(f"Using: {type(embed_model).__name__}")

## Step 4 — Build Vector Index

Embed chunks and store in ChromaDB.

In [ ]:
# ── EXERCISE ──────────────────────────────────────────────────────────────
# Build a ChromaDB index from the nodes.

# YOUR CODE HERE
from src.retrieval.store import get_or_create_index

index = ???
print("Index ready!")

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────
from src.retrieval.store import get_or_create_index

index = get_or_create_index(nodes=nodes, embed_model=embed_model, force_rebuild=True)
print(f"Index ready with {len(nodes)} nodes!")